In [0]:
# ============================================================
# CREATE GOLD LAYER
# ============================================================

CATALOG = "hive_metastore"
SCHEMA_GOLD = "gold"

spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}
""")

print("Gold schema created.")

In [0]:
# ============================================================
# LOAD SILVER TABLES
# ============================================================

SCHEMA_SILVER = "silver"

games_df = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.game_events")

player_metrics_df = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.player_metrics")

player_moves_df = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.player_moves")

display(games_df)
display(player_metrics_df)
display(player_moves_df)

In [0]:
# ============================================================
# PLAYER RESULTS
#
# Creates one record per player per game.
# ============================================================

from pyspark.sql import functions as F

white_results = (
    games_df
    .select(
        "game_id",
        F.col("white_player").alias("player_name"),
        F.when(F.col("winner") == "white", 1).otherwise(0).alias("wins"),
        F.when(F.col("winner") == "black", 1).otherwise(0).alias("losses"),
        F.when(F.col("winner").isNull(), 1).otherwise(0).alias("draws")
    )
)

black_results = (
    games_df
    .select(
        "game_id",
        F.col("black_player").alias("player_name"),
        F.when(F.col("winner") == "black", 1).otherwise(0).alias("wins"),
        F.when(F.col("winner") == "white", 1).otherwise(0).alias("losses"),
        F.when(F.col("winner").isNull(), 1).otherwise(0).alias("draws")
    )
)

player_results_df = white_results.unionByName(black_results)

display(player_results_df)

In [0]:
# ============================================================
# PLAYER RESULTS SUMMARY
# ============================================================

player_results_summary_df = (
    player_results_df
    .groupBy("player_name")
    .agg(
        F.count("*").alias("games_played"),
        F.sum("wins").alias("wins"),
        F.sum("losses").alias("losses"),
        F.sum("draws").alias("draws")
    )
    .withColumn(
        "win_percentage",
        F.round(
            F.col("wins") * 100 / F.col("games_played"),
            2
        )
    )
)

display(player_results_summary_df)

In [0]:
# ============================================================
# GOLD PLAYER LEADERBOARD
# ============================================================

gold_player_leaderboard_df = (
    player_results_summary_df.alias("r")
    .join(
        player_metrics_df.alias("m"),
        "player_name"
    )
    .select(
        "player_name",
        F.col("r.games_played").alias("games_played"),
        "wins",
        "losses",
        "draws",
        "win_percentage",
        "total_moves",
        "avg_think_time_secs",
        "max_think_time_secs",
        "total_clock_loss_secs",
        "avg_remaining_clock_secs"
    )
)

display(gold_player_leaderboard_df)

In [0]:
from pyspark.sql import functions as F

game_metrics_df = (

    player_moves_df

    .groupBy("game_id")

    .agg(

        F.max("move_seq").alias("total_moves"),

        F.avg("think_time_secs").alias("avg_think_time_secs"),

        F.max("think_time_secs").alias("max_think_time_secs"),

        F.sum("clock_loss_secs").alias("total_clock_loss_secs"),

        F.avg("remaining_clock_secs").alias("avg_remaining_clock_secs"),

        F.max("ingested_at").alias("last_update")

    )

)

display(game_metrics_df)

In [0]:
moves_df = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.move_events")

from pyspark.sql.window import Window

latest_window = (
    Window
    .partitionBy("game_id")
    .orderBy(F.desc("move_seq"))
)

latest_position_df = (

    moves_df

    .withColumn(
        "rn",
        F.row_number().over(latest_window)
    )

    .filter("rn = 1")

    .select(

        "game_id",

        F.col("move_seq").alias("current_move_number"),

        "fen",

        "last_move"

    )

)

display(latest_position_df)

In [0]:
from pyspark.sql import functions as F

gold_game_dashboard_df = (

    games_df.alias("g")

    .join(
        game_metrics_df.alias("m"),
        "game_id",
        "left"
    )

    .join(
        latest_position_df.alias("p"),
        "game_id",
        "left"
    )

    .select(

        F.col("game_id"),

        F.col("white_player"),

        F.col("black_player"),

        F.concat_ws(
            " vs ",
            F.col("white_player"),
            F.col("black_player")
        ).alias("player_vs_player"),

        F.col("white_rating"),

        F.col("black_rating"),

        (
            F.abs(
                F.col("white_rating") - F.col("black_rating")
            )
        ).alias("rating_difference"),

        F.col("time_control_seconds"),

        F.col("total_moves"),

        F.round(
            F.col("avg_think_time_secs"),
            2
        ).alias("avg_think_time_secs"),

        F.col("max_think_time_secs"),

        F.col("total_clock_loss_secs"),

        F.round(
            F.col("avg_remaining_clock_secs"),
            2
        ).alias("avg_remaining_clock_secs"),

        F.col("current_move_number"),

        F.col("last_move"),

        F.col("fen").alias("latest_fen"),

        F.col("last_update")

    )

)

display(gold_game_dashboard_df)

In [0]:
print("Games:", gold_game_dashboard_df.count())

display(
    gold_game_dashboard_df
        .groupBy("game_id")
        .count()
        .filter("count > 1")
)

gold_game_dashboard_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in gold_game_dashboard_df.columns
    ]
).display()

In [0]:
GOLD_GAME_DASHBOARD = "gold.live_game_dashboard"

(
    gold_game_dashboard_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(GOLD_GAME_DASHBOARD)
)

print(f"Written to {GOLD_GAME_DASHBOARD}")

In [0]:
display(
    spark.table(GOLD_GAME_DASHBOARD)
)